# 04 — Konversi waveform HDF5 → Parquet + validasi (Bab 2.3, 4.1)

**Kenapa dikonversi?** Spark tidak punya pembaca HDF5. Setelah waveform ada di Parquet, semua pemrosesan berikutnya (A2: ekstraksi fitur, ML) bisa murni Spark: paralel, per kolom, dan bisa di-join dengan metadata.

**Cara kerja** (`src/convert_waveforms.py`):
1. **Driver** membuka HDF5 dengan `h5py` lewat WebHDFS, tapi **hanya membaca layout-nya**: posisi byte (offset), shape, dan dtype tiap bucket. Semua bucket tersimpan *contiguous* tanpa kompresi, jadi trace ke-*i* = potongan byte `[offset + i × ukuran_trace, + ukuran_trace)`.
2. **Metadata bersih** (notebook 03) menentukan trace mana yang dikonversi. Orphan dan baris yang dibuang saat cleaning otomatis tidak ikut.
3. **Spark** membagi daftar trace menjadi potongan berisi ±128 trace berdekatan. Setiap *task* mengambil potongannya dari HDFS dengan **1 permintaan HTTP range** (WebHDFS), memotongnya menjadi trace, lalu mengirimnya ke Spark sebagai batch Arrow (`mapInPandas`).
4. Ditulis sebagai **Parquet dipartisi `dataset`** dengan *dynamic overwrite*: menulis ulang satu dataset tidak menyentuh dataset lain.

Skema tabel `waveforms`:

| Kolom | Tipe | Isi |
|---|---|---|
| `dataset` | string (partisi) | comcat / exotic / noise |
| `trace_name` | string | kunci, bersama `dataset` → join ke metadata |
| `event_id` | string | untuk split train/test per event di A2 (NULL untuk noise) |
| `n_samples` | int | 15001 atau 18001 |
| `n_empty_components` | int | jumlah komponen yang isinya nol semua (dihitung dari waveform, untuk **semua** dataset) |
| `e`, `n`, `z` | array&lt;double&gt; | 3 komponen (urutan ENZ terverifikasi), float64 persis seperti di HDF5 |

Label **tidak** disimpan di tabel ini; label ada di metadata dan diambil lewat join (1 sumber kebenaran untuk label).

Konversi dijalankan dengan:
```bash
caffeinate -i .venv/bin/python src/convert_waveforms.py exotic noise comcat
```
Notebook ini **memvalidasi** hasilnya.

In [1]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath('../src'))
from common import *
import convert_waveforms as cw
import pandas as pd, numpy as np, h5py
from pyspark.sql import functions as F
spark = spark_session('06_validate')
meta = spark.read.parquet(f'{HDFS}{PROCESSED}/metadata')
wf = spark.read.parquet(f'{HDFS}{PROCESSED}/waveforms')
done = sorted(r.dataset for r in wf.select('dataset').distinct().collect())
print('Dataset yang sudah dikonversi:', done)
wf.printSchema()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 23:06:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/23 23:06:26 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Dataset yang sudah dikonversi: ['comcat', 'exotic', 'noise']
root
 |-- trace_name: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- n_samples: integer (nullable = true)
 |-- n_empty_components: integer (nullable = true)
 |-- e: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- n: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- z: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- dataset: string (nullable = true)



## 1. Layout HDF5 yang dipakai konverter

In [2]:
pd.concat([cw.layout(d).assign(dataset=d) for d in DATASETS])

,bucket,offset,bucket_rows,n_samples_h5,dataset
0,bucket1,2048,18790,15001,comcat
1,bucket10,60630567912,18646,15001,comcat
2,bucket2,6764855056,18568,15001,comcat
3,bucket3,13449780688,18545,15001,comcat
4,bucket4,20126425768,18764,15001,comcat
5,bucket5,26881916104,18954,15001,comcat
6,bucket6,33705811000,18497,15001,comcat
7,bucket7,40365174928,18751,15001,comcat
8,bucket8,47115987000,18794,15001,comcat
9,bucket9,53882278056,18744,15001,comcat


## 2. Waktu & throughput konversi

In [3]:
stats = pd.DataFrame(json.load(open(cw.STATS)).values()).set_index('dataset').reindex(DATASETS)
stats

,traces,spark_tasks,seconds,waveform_gb,throughput_mb_s,finished_at
dataset,,,,,,
comcat,178839,88,284.3,64.39,226.5,2026-09-23T23:04:50Z
exotic,8912,5,20.3,3.85,189.9,2026-09-23T22:53:02Z
noise,50227,25,85.2,18.08,212.2,2026-09-23T23:00:05Z


## 3. Validasi 1: jumlah baris Parquet == jumlah baris metadata bersih

In [4]:
cnt = pd.DataFrame({'metadata_clean': meta.groupBy('dataset').count().toPandas().set_index('dataset')['count'],
                    'waveforms_parquet': wf.groupBy('dataset').count().toPandas().set_index('dataset')['count']}).reindex(DATASETS)
cnt['match'] = cnt.metadata_clean == cnt.waveforms_parquet
display(cnt)
for d in done:
    assert cnt.loc[d, 'match'], d
    assert wf.filter(F.col('dataset') == d).select('dataset', 'trace_name').join(meta.select('dataset', 'trace_name'), ['dataset', 'trace_name'], 'left_anti').count() == 0
print('OK: setiap baris waveform punya pasangan metadata, jumlahnya sama')

,metadata_clean,waveforms_parquet,match
dataset,,,
comcat,178839,178839,True
exotic,8912,8912,True
noise,50227,50227,True


OK: setiap baris waveform punya pasangan metadata, jumlahnya sama


## 4. Validasi 2: 100 trace acak per dataset **identik persis** dengan HDF5
Trace diambil acak dari Parquet, lalu dibaca ulang dari HDF5 asli di raw zone dengan `h5py` (jalur baca berbeda dari konverter), dan dibandingkan dengan `np.array_equal` (float64, bit per bit).

In [5]:
fs = webhdfs(); res = []
for d in done:
    names = [r.trace_name for r in wf.filter(F.col('dataset') == d).select('trace_name').orderBy(F.rand(42)).limit(100).collect()]
    sample = (wf.filter((F.col('dataset') == d) & F.col('trace_name').isin(names))
                .join(F.broadcast(meta.select('dataset', 'trace_name', 'bucket', 'row')), ['dataset', 'trace_name']).collect())
    with fs.open(f'{RAW}/{d}/{d}_waveforms.hdf5', 'rb') as fo, h5py.File(fo, 'r') as f:
        ok = sum(np.array_equal(np.stack([r.e, r.n, r.z]), f['data'][r.bucket][r.row]) for r in sample)
    res.append({'dataset': d, 'checked': len(sample), 'identical': ok})
res = pd.DataFrame(res); display(res)
assert (res.checked == res.identical).all() and (res.checked == 100).all()

,dataset,checked,identical
0,comcat,100,100
1,exotic,100,100
2,noise,100,100


## 5. Ukuran: HDF5 (raw) vs Parquet (processed)
Parquet memakai kompresi **Snappy** per kolom. Angka seismik float64 di sini sebenarnya nilai *count* sensor (bilangan bulat), sehingga banyak byte-nya berulang dan cukup bisa dikompres.

In [6]:
def du(path):
    out = sh(f'hdfs dfs -du -s {path}', quiet=True).split()
    return int(out[0])
size = []
for d in DATASETS:
    raw_b = du(f'{RAW}/{d}/{d}_waveforms.hdf5')
    pq_b = du(f'{PROCESSED}/waveforms/dataset={d}') if d in done else None
    size.append({'dataset': d, 'hdf5_raw_GB': round(raw_b / 1e9, 2),
                 'parquet_GB': round(pq_b / 1e9, 2) if pq_b else None,
                 'parquet_vs_hdf5_%': round(pq_b / raw_b * 100, 1) if pq_b else None})
size = pd.DataFrame(size); size

,dataset,hdf5_raw_GB,parquet_GB,parquet_vs_hdf5_%
0,comcat,67.34,53.73,79.8
1,exotic,4.11,2.54,61.7
2,noise,18.53,15.11,81.5


In [7]:
sh(f'hdfs dfs -ls {PROCESSED}/waveforms/dataset=exotic')
sh(f'hdfs fsck {PROCESSED}/waveforms -files -blocks | tail -20')

$ hdfs dfs -ls /datalake/processed/waveforms/dataset=exotic
Found 5 items
-rw-r--r--   1 axelnakata supergroup  591790191 2026-09-23 22:53 /datalake/processed/waveforms/dataset=exotic/part-00000-ad7884e8-6661-41b9-ba41-ef9f5ce94030.c000.snappy.parquet
-rw-r--r--   1 axelnakata supergroup  577349655 2026-09-23 22:53 /datalake/processed/waveforms/dataset=exotic/part-00001-ad7884e8-6661-41b9-ba41-ef9f5ce94030.c000.snappy.parquet
-rw-r--r--   1 axelnakata supergroup  224141731 2026-09-23 22:52 /datalake/processed/waveforms/dataset=exotic/part-00002-ad7884e8-6661-41b9-ba41-ef9f5ce94030.c000.snappy.parquet
-rw-r--r--   1 axelnakata supergroup  548261552 2026-09-23 22:53 /datalake/processed/waveforms/dataset=exotic/part-00003-ad7884e8-6661-41b9-ba41-ef9f5ce94030.c000.snappy.parquet
-rw-r--r--   1 axelnakata supergroup  595230267 2026-09-23 22:53 /datalake/processed/waveforms/dataset=exotic/part-00004-ad7884e8-6661-41b9-ba41-ef9f5ce94030.c000.snappy.parquet


$ hdfs fsck /datalake/processed/waveforms -files -blocks | tail -20
Missing replicas:		0 (0.0 %)
 Blocks queued for replication:	0

Erasure Coded Block Groups:
 Total size:	0 B
 Total files:	0
 Total block groups (validated):	0
 Minimally erasure-coded block groups:	0
 Over-erasure-coded block groups:	0
 Under-erasure-coded block groups:	0
 Unsatisfactory placement block groups:	0
 Average block group size:	0.0
 Missing block groups:		0
 Corrupt block groups:		0
 Missing internal blocks:	0
 Blocks queued for replication:	0
FSCK ended at Thu Sep 24 06:07:24 WIB 2026 in 7 milliseconds


The filesystem under path '/datalake/processed/waveforms' is HEALTHY
Connecting to namenode via http://localhost:9870/fsck?ugi=axelnakata&files=1&blocks=1&path=%2Fdatalake%2Fprocessed%2Fwaveforms


"Missing replicas:\t\t0 (0.0 %)\n Blocks queued for replication:\t0\n\nErasure Coded Block Groups:\n Total size:\t0 B\n Total files:\t0\n Total block groups (validated):\t0\n Minimally erasure-coded block groups:\t0\n Over-erasure-coded block groups:\t0\n Under-erasure-coded block groups:\t0\n Unsatisfactory placement block groups:\t0\n Average block group size:\t0.0\n Missing block groups:\t\t0\n Corrupt block groups:\t\t0\n Missing internal blocks:\t0\n Blocks queued for replication:\t0\nFSCK ended at Thu Sep 24 06:07:24 WIB 2026 in 7 milliseconds\n\n\nThe filesystem under path '/datalake/processed/waveforms' is HEALTHY\nConnecting to namenode via http://localhost:9870/fsck?ugi=axelnakata&files=1&blocks=1&path=%2Fdatalake%2Fprocessed%2Fwaveforms"

## 6. Komponen kosong, dihitung dari waveform untuk semua dataset
Di notebook 02 (sampel 300) terlihat Noise lebih sering punya komponen kosong. Sekarang dihitung dari **semua** trace. Untuk ComCat, hasilnya dibandingkan dengan kolom resmi `trace_missing_channel`.

In [8]:
j = wf.select('dataset', 'trace_name', 'n_empty_components').join(
        F.broadcast(meta).select('dataset', 'trace_name', 'station_channel_code', 'label', 'trace_missing_channel'), ['dataset', 'trace_name'])
emp = j.groupBy('label', 'station_channel_code').pivot('n_empty_components').count().fillna(0).orderBy('label', 'station_channel_code').toPandas()
emp

,label,station_channel_code,0,1,2
0,earthquake,BH,49227,26,3
1,earthquake,EH,41063,3128,46161
2,earthquake,HH,22828,190,74
3,explosion,BH,4737,2,2
4,explosion,EH,4181,160,3646
5,explosion,HH,3350,54,7
6,noise,BH,11251,294,4295
7,noise,EH,17850,579,8516
8,noise,HH,6103,116,1223
9,surface_event,BH,1604,0,1


In [9]:
if 'comcat' in done:
    agree = j.filter("dataset='comcat'").groupBy((F.col('n_empty_components') == F.col('trace_missing_channel')).alias('agree')).count().toPandas()
    display(agree)

,agree,count
0,True,178839


## 7. Contoh isi tabel

In [10]:
wf.select('dataset', 'trace_name', 'event_id', 'n_samples', 'n_empty_components',
          F.size('z').alias('len_z'), F.slice('z', 7000, 5).alias('z[7000:7005]')).show(5, truncate=False)

+-------+-----------------------+----------+---------+------------------+-----+--------------------------------------------------------------------------------------------------------+
|dataset|trace_name             |event_id  |n_samples|n_empty_components|len_z|z[7000:7005]                                                                                            |
+-------+-----------------------+----------+---------+------------------+-----+--------------------------------------------------------------------------------------------------------+
|comcat |bucket1$10752,:3,:15001|uw61129597|15001    |0                 |15001|[438.7053771008587, 430.20730271446917, 430.70015126991746, 442.97609983105855, 466.0675088536968]      |
|comcat |bucket1$10753,:3,:15001|uw61129532|15001    |2                 |15001|[-15.168854424071737, -25.917758342968853, -25.91511602176865, -18.411554988401434, -15.158723075086485]|
|comcat |bucket1$10754,:3,:15001|uw61129427|15001    |0                 |15

## 8. Ringkasan untuk laporan

In [11]:
summary = cnt.join(stats[['seconds', 'throughput_mb_s', 'spark_tasks']]).join(size.set_index('dataset')).reset_index(names='dataset')
summary = summary.merge(res, on='dataset', how='left')
save_md(summary, 'waveform_conversion', 'Konversi waveform HDF5 → Parquet',
        'Validasi: jumlah baris = metadata bersih; 100 trace acak per dataset identik bit-per-bit dengan HDF5 (np.array_equal).')
save_md(emp, 'empty_components', 'Jumlah komponen kosong per kelas × tipe sensor (dari waveform)')
summary

,dataset,metadata_clean,waveforms_parquet,match,seconds,throughput_mb_s,spark_tasks,hdf5_raw_GB,parquet_GB,parquet_vs_hdf5_%,checked,identical
0,comcat,178839,178839,True,284.3,226.5,88,67.34,53.73,79.8,100,100
1,exotic,8912,8912,True,20.3,189.9,5,4.11,2.54,61.7,100,100
2,noise,50227,50227,True,85.2,212.2,25,18.53,15.11,81.5,100,100


In [12]:
spark.stop()